# Gold Layer Transformations from Silver

## Funding Gap & Ranking Methodology

### Funding Gap (Table 1 — `njyoti_gold_country_year_funding`)

Gap metrics are computed at the **country × year** grain from FTS appeals data:

| Column | Formula | Notes |
|---|---|---|
| `funding_gap_usd` | `requirements − funding` | Absolute shortfall in USD |
| `funding_gap_pct` | `(1 − funding/requirements) × 100` | % of need unmet |
| `fts_percent_funded` | `funding / requirements × 100` | Inverse of gap; primary threshold signal |
| `years_below_threshold` | Rolling 5-year count of years where `fts_percent_funded < 33%` | Chronic underfunding persistence signal |
| `yoy_pct_funded_change` | `fts_percent_funded(t) − fts_percent_funded(t−1)` | Year-over-year trajectory; negative = worsening |

**Threshold calibration:** 33% was chosen empirically (P25–P33 band of the `fts_percent_funded` distribution). At 50%, nearly all 2023+ country-years qualified, collapsing discriminatory power. At 33%, ~28% of rows qualify — capturing only the most critically underfunded contexts.

---

### Overlooked Crisis Score (Table 3 — `njyoti_gold_overlooked_crises_scored`)

The composite score is **multiplicative** — every factor must be non-zero for a high rank. A well-funded crisis scores 0 regardless of severity; a low-severity crisis scores low regardless of funding gap.

```
overlooked_score =  severity_norm
                  × funding_gap_norm
                  × scale_norm
                  × trend_multiplier
                  × persistence_multiplier
                  × allocation_neglect
```

| Component | Formula | Range | Interpretation |
|---|---|---|---|
| `severity_norm` | `inform_severity_index / 5.0` | [0, 1] | INFORM index normalised to 1–5 scale (2026 values ÷2 in Table 1) |
| `funding_gap_norm` | `CLAMP(1 − fts_percent_funded/100, 0, 1)` | [0, 1] | 0% funded → 1.0; fully funded → 0.0 |
| `scale_norm` | `CLAMP((log₁₀(requirements) − 5) / 5, 0, 1)` | [0, 1] | Normalised log-scale of USD requirements; rewards larger crises |
| `trend_multiplier` | Lookup: Increasing=1.3, Stable=1.0, Decreasing=0.8, −/NULL=1.0 | [0.8, 1.3] | OCHA-validated severity trajectory |
| `persistence_multiplier` | `1.0 + (years_below_threshold / 5) × 0.5` | [1.0, 1.5] | Rewards chronic underfunding (up to 5 years lookback) |
| `allocation_neglect` | `1 − MIN(1, allocations / requirements)` | [0, 1] | 0 CERF/CBPF allocations → 1.0 (fully neglected); full coverage → 0.0 |

### Ranking

`rank_in_year` is computed as `DENSE_RANK()` partitioned by `year`, ordered by `overlooked_score DESC`. Rank 1 = most overlooked crisis in that year.

---

### Sector-Level Gap (Table 2 — `njyoti_gold_sector_funding_gaps`)

Parallel gap metrics at **country × year × sector_canonical** grain, sourced from HNO (needs) joined to FTS cluster funding via canonical sector mapping:

| Column | Formula |
|---|---|
| `sector_funding_gap_usd` | `sector_requirements_usd − COALESCE(sector_funding_usd, 0)` |
| `sector_pct_funded` | `sector_funding_usd / sector_requirements_usd × 100` |
| `coverage_ratio` | `targeted / in_need` (people targeted ÷ people in need) |

Sector match rate: **95.4%** via OCHA taxonomy bridge join (up from 67% with regex matching). All 19 HNO sectors including Protection sub-clusters (Child Protection, GBV, Mine Action, HLP) now have FTS funding data.

---

### Key Assumptions

- **A1:** FTS requirements = stated need (politically constructed; some over/under-appeal).
- **A3:** Absence from INFORM ≠ absence of crisis. 17 countries score via FTS-only data; `country_name` resolved from C&T Taxonomy MVP.
- **A6:** `fts_percent_funded` captures only same-plan funding; bilateral aid outside FTS is excluded.

---

## Limitations

### Scoring Universe
- **43.2% of FTS requirements are unscored.** Countries with FTS funding data but no INFORM severity index cannot be ranked. These are not scored zero — they are simply absent from Table 3. Agents should surface this caveat when answering "which crises are most overlooked."
- **Pre-2020 reliability is low.** INFORM coverage before 2020 is sparse. `severity_norm` will be NULL for most pre-2020 rows, meaning scoring is only reliable for 2020+.
- **HNO sector needs cover 2024–2026 only** and 24 countries. Table 2 (`sector_funding_gaps`) is not usable for multi-year trend analysis or for countries outside this set.

### Severity Data
- **2026 scale change (÷2 normalization).** OCHA expanded the INFORM index from 1–5 to 1–10 in January 2026. All 2026 values are halved at Table 1 ingestion. This is a best-effort normalisation — if OCHA revises the methodology further, this fix must be updated.
- **INFORM trend label is country-level, not crisis-specific.** For countries with multiple active crises (e.g., COD has 7), the "worst" trend label is taken. A country with one improving and one worsening crisis will carry `Increasing` even if the larger crisis is stable.
- **`severity_year_delta` can be noisy** for years with only one snapshot (delta = 0 by construction).

### Funding Data
- **Bilateral aid outside FTS is excluded** (Assumption A6). `fts_percent_funded` understates true funding for countries with large bilateral donor relationships (e.g., some middle-income crisis contexts).
- **Overfunding artefacts:** 39 country-years show `funding_gap_usd < 0` (more funding than requirements). This occurs when carry-over funding from prior plans is booked against a new year's requirements. These are real FTS data patterns, not transformation errors.
- **Allocations coverage is 35%** of scored rows. The `allocation_neglect` component defaults to 1.0 (fully neglected) for the 65% with no CERF/CBPF data, which may overstate neglect for countries with other response mechanisms.

### Population
- **Static population denominator.** COD population is joined on `iso3` only (no year), giving one figure per country applied across all years. Coverage is 101/126 countries. Per-capita metrics (`per_capita_funding_usd`, `per_capita_requirements_usd`) are unreliable for countries missing COD data.

---

## Known Issues

| ID | Table | Issue | Status |
|---|---|---|---|
| C1 | T3 | 17 iso3 codes in scorecard absent from INFORM registry | Resolved — all confirmed in C&T Taxonomy. Validation check updated to use union of registry + taxonomy. Informational note printed at runtime. |
| B3-sub | T2 | Protection sub-cluster match was 0% with regex approach | RESOLVED — 95.4% match via taxonomy join. Cell 6 refactored to use hpc_global_cluster_taxonomy bridge table. All sub-clusters now matched (Child Protection 100%, GBV 94.1%, Mine Action 87.5%, HLP 79.2%). |
| B3-WASH | T2 | WASH sector match rate was 57% with regex approach | RESOLVED — 98.2% match via taxonomy join. |
| A5 | T1 | COD population filter uses `population_group="T_TL"`, not `"all"` | Fixed. Original filter returned 0 matches. Current filter gives 82% coverage. |
| scale | T1, T5 | 2026 INFORM severity on 1–10 scale instead of 1–5 | Fixed. `/2` normalization applied at Table 1 and Table 5 ingest. Must re-verify if OCHA releases a 2027 methodology update. |
| schema | T2 | `overwriteSchema=True` required on Table 2 write | Fixed. Added after B3 canonical mapping introduced new columns (`sector_canonical`, `fts_cluster_names_matched`). |
| C3-T5 | T5 | 251,008 → 6,889 rows after deduplication | Fixed. Each snapshot file contained full history; deduplicated to one row per `crisis_id × year_month`, preferring pre-2026 snapshots. |
| HNO-fanout | T2 | `in_need` and `targeted` overcounted 2–12× for 70% of sector-country-year rows. HNO stores national totals AND all population sub-group breakdowns (gender × age × displacement type) at national level with all admin pcodes NULL. Produced impossible `coverage_ratio > 1.0` for 3 rows and `targeting_gap_people < 0` for several. | Fixed. Added `category IN ('total', NULL)` filter to keep only national aggregate rows, followed by `ROW_NUMBER()` dedup (highest `in_need` per `iso3 × year × cluster`) to resolve residual description-level duplicates. Zero data loss — all iso3+year+sector groups have at least one total/NULL row. FTS columns (`sector_requirements_usd`, `sector_funding_usd`, `sector_pct_funded`) unaffected. |

---

## Failure Cases & Edge Behaviour

- **Score = 0:** 19 rows in Table 3 score exactly 0. This occurs when `allocation_neglect = 0` (fully allocated crisis) or `funding_gap_norm = 0` (fully funded). These are correct — a well-funded or fully-noticed crisis is not overlooked by design.
- **`rank_in_year = 1` ties:** `DENSE_RANK()` allows ties. Two countries with identical scores in the same year will both hold rank 1. Downstream consumers should not assume rank 1 is unique.
- **`persist_multiplier` at minimum (1.0)** for all countries with fewer than 5 years of FTS history. New crises or newly-appealing countries are not penalised, but they also gain no persistence boost.
- **PRK (North Korea) scores ~0.55 (top-5 in 2026) but is `inform_tracked = False`.** Its severity is derived from FTS cross-snapshot data only. The score is directionally meaningful but lacks the INFORM validation that underpins other top-ranked countries. RAG agents must caveat this.
- **Americas structural pattern:** 16/22 years show < 50% funding. This is partly a reporting artefact — Venezuela and Haiti dominate the region and are chronically underfunded. Regional summary figures for Americas should not be interpreted as representative of the full continent.

In [0]:
# =============================================================================
# Gold Layer — UNOCHA Humanitarian Funding Gap Analysis Pipeline
# =============================================================================
# Catalog: workspace | Schema: default | Prefix: njyoti_gold_
#
# PURPOSE: Enable identification of "overlooked" crises — situations where
# there is a significant mismatch between human need and actual funding.
#
# GOLD TABLES:
#   1. njyoti_gold_country_year_funding    — Country×Year fact (the workhorse)
#   2. njyoti_gold_sector_funding_gaps     — Sector-level needs + actual funding
#   3. njyoti_gold_overlooked_crises_scored — Composite scorecard with rankings
#   4. njyoti_gold_regional_summary        — Regional aggregations for trends
#   5. njyoti_gold_inform_temporal_series   — Crisis severity time series (long)
#
# SILVER TABLES USED (14 of 23):
#   - fts_requirements_funding_global       → funding gap (core)
#   - fts_requirements_funding_globalcluster_global → sector-level funding (via taxonomy join)
#   - hpc_global_cluster_taxonomy            → sector bridge (cluster_code ↔ cluster_id ↔ canonical name)
#   - humanitarian_response_plans           → plan requirements (exploded locations)
#   - inform_all_crises                     → severity index/category
#   - inform_trends                         → severity trajectory over time
#   - inform_crisis_registry                → ISO3 bridge, region, drivers
#   - cod_population                        → per-capita normalization
#   - allocations                           → CERF/CBPF system response
#   - hpc_hno                               → sector-level needs (people)
#   - contributions                         → macro context (global pool trends)
#   - country_territory_taxonomy            → iso3→country_name authoritative fallback
#                                              (C&T Taxonomy MVP; covers 17 orphan iso3
#                                               codes absent from INFORM registry)
#
# SILVER TABLES EXCLUDED (with rationale):
#   - fts_incoming/outgoing/internal_funding → redundant with aggregate FTS
#   - fts_requirements_funding_covid_global  → COVID-specific, not general gap
#   - fts_requirements_funding_cluster_global → replaced by globalcluster_global + taxonomy join
#   - fts_requirements_funding_globalcluster → global-level (no country grain)
#   - inform_complexity/conditions/impact    → sub-dimension detail (optional)
#   - inform_indicator_bounds               → reference metadata only
#   - sentinel_log / audit_log              → pipeline ops, not analytical
#
# =============================================================================
# ASSUMPTIONS (must be stated for defensibility):
# =============================================================================
# A1. FTS requirements = stated need. These are politically constructed appeals,
#     not ground truth. Some countries over-appeal, others under-appeal.
# A2. INFORM severity is the best-available crisis intensity proxy (index 0-5).
# A3. Absence of INFORM tracking ≠ absence of crisis. Untracked crises are
#     invisible to the ranking — a known blindspot surfaced in analysis narrative.
# A4. "Overlooked" = underfunded AND lacking institutional response (allocations).
#     A well-funded crisis is not overlooked; an allocated-to crisis is noticed.
# A5. COD population: we filter to population_group="all", gender="all", no admin
#     codes, to get national totals without double-counting age bracket overlaps.
# A6. FTS percent_funded is computed from requirements & funding in the SAME plan.
#     Cross-plan funding (bilateral aid outside FTS) is NOT captured.
# A7. INFORM trend label (Stable/Increasing/Decreasing) is OCHA-validated.
#     "-" means under-monitored, not "no trend" — a meta-signal of neglect.
# A8. Allocations → ISO3 mapping via crisis_registry is 100% (32/32 validated).
# A9. HRP locations are pipe-delimited ISO3 codes; EXPLODE+TRIM for joins.
# A10. Year overlap: HNO covers 2024-2026 only. Scoring is most reliable for
#      years with multi-source coverage (2020+). Pre-2020 has FTS only.
# A11. INFORM within-year delta (severity_year_delta): positive = worsening
#      within the funding cycle. A crisis at 2.0 in Jan that reaches 4.5 by Sep
#      signals escalation that may have been missed by early-year funding decisions.
# =============================================================================

CATALOG = "workspace"
SCHEMA = "default"
GOLD_PREFIX = "njyoti_gold_"
SILVER_PREFIX = "njyoti_silver_unocha_"

# =============================================================================
# CONFIGURABLE PARAMETERS
# =============================================================================
# UNDERFUNDED_THRESHOLD: Calibrated from empirical distribution analysis.
#
# CALIBRATION DECISION LOG (performed on Gold Table 1 output):
#   - Initial value: 50%
#   - Distribution shows global median at 51.2% (P50), P25 at 31.2%
#   - 48.5% of all country-years fall below 50%
#   - PROBLEM: Year-by-year medians show structural decline:
#       2019: 52.6%, 2020: 41.9%, 2023: 35%, 2024: 40%, 2025: 23%, 2026: 17%
#   - At 50%, nearly ALL recent country-years (2023+) qualify as "underfunded"
#     which collapses the persistence multiplier's discriminatory power.
#   - DECISION: Lower to 33% (approximately P25-P33 band).
#     At 33%, ~28% of rows qualify — captures only the bottom third of
#     historical funding. This means "chronically critically underfunded"
#     rather than "chronically below average" — a stronger, more defensible
#     signal for identifying truly overlooked crises.
#   - The continuous funding_gap_norm (0-1) in the score formula still
#     captures the full spectrum; persistence is reserved for severe gaps.
# =============================================================================
UNDERFUNDED_THRESHOLD = 33  # % funded below which a crisis is considered critically underfunded
PERSISTENCE_LOOKBACK_YEARS = 5  # years of history for chronic underfunding signal
TREND_MULTIPLIER_MAP = {
    "Increasing": 1.3,   # crisis worsening → more overlooked
    "Stable": 1.0,       # no change
    "Decreasing": 0.8,   # crisis improving → less overlooked
    "-": 1.0,            # under-monitored (no data, neutral)
    None: 1.0            # missing
}

print("Configuration loaded.")
print(f"  UNDERFUNDED_THRESHOLD: {UNDERFUNDED_THRESHOLD}% (calibrated from P25-P33 band)")
print(f"  PERSISTENCE_LOOKBACK: {PERSISTENCE_LOOKBACK_YEARS} years")
print(f"  Trend multipliers: {TREND_MULTIPLIER_MAP}")

In [0]:
# =============================================================================
# GOLD TABLE 1: Country-Year Funding Fact Table
# Grain: iso3 × year (one row per country per year)
# =============================================================================
# This is the WORKHORSE table. Most analytical queries start here.
# Sources: FTS + HRP + INFORM severity + INFORM trends + COD pop + Allocations
# =============================================================================

from pyspark.sql import functions as F
from pyspark.sql.window import Window

# --- Load all source tables ---
fts_req = spark.table(f"{CATALOG}.{SCHEMA}.{SILVER_PREFIX}fts_requirements_funding_global")
hrp = spark.table(f"{CATALOG}.{SCHEMA}.{SILVER_PREFIX}humanitarian_response_plans")
inform = spark.table(f"{CATALOG}.{SCHEMA}.{SILVER_PREFIX}inform_all_crises")
inform_trends = spark.table(f"{CATALOG}.{SCHEMA}.{SILVER_PREFIX}inform_trends")
cod_pop = spark.table(f"{CATALOG}.{SCHEMA}.{SILVER_PREFIX}cod_population")
allocations = spark.table(f"{CATALOG}.{SCHEMA}.{SILVER_PREFIX}allocations")
registry = spark.table(f"{CATALOG}.{SCHEMA}.{SILVER_PREFIX}inform_crisis_registry")

# =============================================================================
# 1a. FTS: Aggregate requirements & funding by country+year
# R4: Also collect plan codes for provenance tracing
# =============================================================================
fts_country_year = (
    fts_req
    .filter(F.col("requirements").isNotNull())
    .groupBy("countrycode", "year")
    .agg(
        F.sum("requirements").alias("fts_requirements_usd"),
        F.sum("funding").alias("fts_funding_usd"),
        F.count("*").alias("fts_plan_count"),
        # R4: Provenance — which plan codes contributed to this row
        F.concat_ws(", ", F.collect_set("code")).alias("fts_plan_codes")
    )
    .withColumn(
        "fts_percent_funded",
        F.when(F.col("fts_requirements_usd") > 0,
               F.round(F.col("fts_funding_usd") / F.col("fts_requirements_usd") * 100, 1))
    )
    .withColumnRenamed("countrycode", "iso3")
)

# --- Temporal features on FTS (persistence + YoY change) ---
w_history = Window.partitionBy("iso3").orderBy("year").rowsBetween(-PERSISTENCE_LOOKBACK_YEARS + 1, 0)
w_lag = Window.partitionBy("iso3").orderBy("year")

fts_country_year = (
    fts_country_year
    # Persistence: count of years in lookback window where funding < threshold
    .withColumn(
        "years_below_threshold",
        F.sum(
            F.when(F.col("fts_percent_funded") < UNDERFUNDED_THRESHOLD, 1).otherwise(0)
        ).over(w_history)
    )
    # YoY change in percent funded (positive = improving, negative = worsening)
    .withColumn(
        "yoy_pct_funded_change",
        F.round(F.col("fts_percent_funded") - F.lag("fts_percent_funded", 1).over(w_lag), 1)
    )
)

# =============================================================================
# 1b. HRP: Explode pipe-delimited locations, aggregate by country+year
# =============================================================================
hrp_exploded = (
    hrp
    .filter(F.col("locations").isNotNull())
    .withColumn("iso3", F.explode(F.split(F.col("locations"), "\\|")))
    .withColumn("iso3", F.trim(F.col("iso3")))
    .filter(F.length(F.col("iso3")) == 3)
    .groupBy("iso3", "year")
    .agg(
        F.sum("revised_requirements_usd").alias("hrp_revised_requirements_usd"),
        F.sum("orig_requirements_usd").alias("hrp_orig_requirements_usd"),
        F.count("*").alias("hrp_plan_count")
    )
)

# =============================================================================
# 1c. INFORM: Severity per country-year (latest + within-year summary)
# Parse year from snapshot_month (format: "september-2025")
#
# We compute THREE temporal signals:
#   - inform_severity_index: latest snapshot value (current state)
#   - severity_year_avg: average across all snapshots in that year (robust)
#   - severity_year_delta: latest minus earliest in year (within-year trajectory)
#     Positive delta = worsening within the year (escalating while unfunded)
#
# SCALE NORMALIZATION:
#   OCHA changed INFORM methodology in Jan 2026: index expanded from 1-5 to
#   1-10 scale. All 63 pre-2026 snapshot files have max_index <= 5.0; all 4
#   2026 snapshots reach up to 9.7. We normalize 2026 values by /2 so that
#   inform_severity_index is ALWAYS on the 1-5 scale in this table.
#   This ensures cross-year comparability and consistent downstream scoring.
# =============================================================================
inform_with_year = (
    inform
    .withColumn("inform_year",
                F.regexp_extract(F.col("snapshot_month"), r"-(\d{4})$", 1).cast("int"))
    .filter(F.col("inform_year").isNotNull())
)

# Latest snapshot per country-year (for current state)
w_inform = Window.partitionBy("iso3", "inform_year").orderBy(F.col("snapshot_month").desc())
w_inform_asc = Window.partitionBy("iso3", "inform_year").orderBy(F.col("snapshot_month").asc())

inform_enriched = (
    inform_with_year
    .withColumn("rn_desc", F.row_number().over(w_inform))  # latest = 1
    .withColumn("rn_asc", F.row_number().over(w_inform_asc))  # earliest = 1
)

# Aggregate: latest value, earliest value, average, count of snapshots
inform_country_year = (
    inform_enriched
    .groupBy("iso3", "inform_year")
    .agg(
        # Latest severity (current state)
        F.max(F.when(F.col("rn_desc") == 1, F.col("inform_severity_index"))).alias("inform_severity_index"),
        F.max(F.when(F.col("rn_desc") == 1, F.col("inform_severity_category"))).alias("inform_severity_category"),
        # Year average (robust central tendency)
        F.round(F.avg("inform_severity_index"), 2).alias("severity_year_avg"),
        # Within-year delta: latest minus earliest (positive = worsening)
        F.round(
            F.max(F.when(F.col("rn_desc") == 1, F.col("inform_severity_index"))) -
            F.max(F.when(F.col("rn_asc") == 1, F.col("inform_severity_index"))),
            2
        ).alias("severity_year_delta"),
        # Count of snapshots in year (data density indicator)
        F.count("*").alias("inform_snapshots_in_year")
    )
    .withColumnRenamed("inform_year", "year")
)

# --- Normalize 2026 severity to 1-5 scale (source uses 1-10 from Jan 2026) ---
inform_country_year = (
    inform_country_year
    .withColumn("inform_severity_index",
                F.when(F.col("year") >= 2026, F.round(F.col("inform_severity_index") / 2.0, 2))
                 .otherwise(F.col("inform_severity_index")))
    .withColumn("severity_year_avg",
                F.when(F.col("year") >= 2026, F.round(F.col("severity_year_avg") / 2.0, 2))
                 .otherwise(F.col("severity_year_avg")))
    .withColumn("severity_year_delta",
                F.when(F.col("year") >= 2026, F.round(F.col("severity_year_delta") / 2.0, 2))
                 .otherwise(F.col("severity_year_delta")))
    # Category stays as-is (always 1-5 regardless of source scale)
)

# =============================================================================
# 1d. INFORM TRENDS: Severity trajectory per country (cross-year)
# Uses pre-computed 'trend' label (OCHA-validated)
# =============================================================================
trends_latest = (
    inform_trends
    .withColumn("rn", F.row_number().over(
        Window.partitionBy("crisis_id").orderBy(F.col("snapshot_month").desc())
    ))
    .filter(F.col("rn") == 1)
    .select("crisis_id", "trend")
)

# For countries with multiple crises, take the "worst" trend
trend_priority = F.when(F.col("trend") == "Increasing", 3) \
    .when(F.col("trend") == "Stable", 2) \
    .when(F.col("trend") == "Decreasing", 1) \
    .otherwise(0)

trends_by_country = (
    trends_latest
    .join(registry.select("crisis_id", "iso3"), on="crisis_id", how="inner")
    .withColumn("trend_priority", trend_priority)
    .withColumn("rn", F.row_number().over(
        Window.partitionBy("iso3").orderBy(F.col("trend_priority").desc())
    ))
    .filter(F.col("rn") == 1)
    .select("iso3", F.col("trend").alias("severity_trend_label"))
)

# =============================================================================
# 1e. COD Population: National total per country (static, applied to all years)
# =============================================================================
# FIX: Original filter used population_group="all" which doesn't exist.
# Correct filter: population_group="T_TL" (Total gender, Total age) + admin_level=0
# COD has exactly 1 national total per country (varying reference_year, 2001-2025).
# We join on iso3 ONLY (not year) since there's just one population figure per country.
# This gives a static denominator for per-capita metrics — imperfect but the only
# option without annual population estimates. Coverage: 101/126 countries.
# ASSUMPTION A5 (updated): population_group="T_TL", gender="all", admin_level=0
cod_national = (
    cod_pop
    .filter(
        (F.col("admin_level") == 0) &
        (F.col("population_group") == "T_TL") &
        (F.col("gender") == "all")
    )
    .groupBy("iso3")
    .agg(F.sum("population").alias("total_population"))
)

# =============================================================================
# 1f. Allocations: Map country_name → ISO3 via crisis_registry
# ASSUMPTION A8: 100% coverage (32/32 countries validated)
# =============================================================================
allocations_mapped = (
    allocations
    .join(
        registry.select("country_name", "iso3").dropDuplicates(["country_name"]),
        on="country_name",
        how="inner"
    )
    .groupBy("iso3", "year")
    .agg(
        F.sum("budget_usd").alias("total_allocations_usd"),
        F.sum(F.when(F.col("allocation_type") == "standard", F.col("budget_usd"))).alias("standard_allocations_usd"),
        F.sum(F.when(F.col("allocation_type") == "reserve", F.col("budget_usd"))).alias("reserve_allocations_usd")
    )
)

# =============================================================================
# 1g. Crisis registry: Region + country_name (one row per iso3)
# Country name resolution (coalesce precedence):
#   1. inform_crisis_registry.country_name  — contextually relevant for active crises
#   2. country_territory_taxonomy.country_name — authoritative fallback for ALL iso3 codes
#      Source: njyoti_silver_unocha_country_territory_taxonomy (C&T Taxonomy MVP)
#      Covers 17 orphan iso3 codes absent from INFORM registry:
#        Group A (crisis-adjacent, INFORM coverage gap): PRK, ZWE, ZMB, COG, BOL,
#          VNM, NPL, RWA, MNG, LSO
#        Group B (Ukraine refugee-hosting EU/Eastern Europe): POL, MDA, CZE, ROU,
#          BGR, LTU, EST
# This coalesce propagates to Table 3 (reads from saved Table 1 Delta) automatically.
# =============================================================================
ct_taxonomy = (
    spark.table(f"{CATALOG}.{SCHEMA}.{SILVER_PREFIX}country_territory_taxonomy")
    .filter(F.col("admin_level") == 0)
    .select(
        F.col("iso3").alias("ct_iso3"),
        F.col("country_name").alias("ct_country_name")
    )
)

registry_country = (
    registry
    .select("iso3", "region", "country_name")
    .dropDuplicates(["iso3"])
    .join(ct_taxonomy, F.col("iso3") == F.col("ct_iso3"), how="left")
    .withColumn("country_name", F.coalesce(F.col("country_name"), F.col("ct_country_name")))
    .drop("ct_iso3", "ct_country_name")
)

# =============================================================================
# 1h. ASSEMBLE: Full outer on FTS+HRP spine, left-join everything else
# =============================================================================
gold_country_year = (
    fts_country_year
    .join(hrp_exploded, on=["iso3", "year"], how="full_outer")
    .join(inform_country_year, on=["iso3", "year"], how="left")
    .join(trends_by_country, on="iso3", how="left")
    .join(cod_national, on="iso3", how="left")  # iso3 only (static population)
    .join(allocations_mapped, on=["iso3", "year"], how="left")
    .join(registry_country, on="iso3", how="left")
    # --- Derived metrics ---
    .withColumn("funding_gap_usd",
                F.coalesce(F.col("fts_requirements_usd"), F.lit(0)) -
                F.coalesce(F.col("fts_funding_usd"), F.lit(0)))
    .withColumn("funding_gap_pct",
                F.when(F.col("fts_requirements_usd") > 0,
                       F.round((1 - F.col("fts_funding_usd") / F.col("fts_requirements_usd")) * 100, 1)))
    .withColumn("allocation_coverage_pct",
                F.when(F.col("fts_requirements_usd") > 0,
                       F.round(F.col("total_allocations_usd") / F.col("fts_requirements_usd") * 100, 2)))
    .withColumn("per_capita_funding_usd",
                F.when(F.col("total_population") > 0,
                       F.round(F.col("fts_funding_usd") / F.col("total_population"), 2)))
    .withColumn("per_capita_requirements_usd",
                F.when(F.col("total_population") > 0,
                       F.round(F.col("fts_requirements_usd") / F.col("total_population"), 2)))
    # --- R2: Data completeness indicator ---
    # Shows which sources contributed to each row (for agent confidence/explainability)
    .withColumn("data_sources",
                F.concat_ws("|",
                    F.when(F.col("fts_requirements_usd").isNotNull(), F.lit("FTS")),
                    F.when(F.col("inform_severity_index").isNotNull(), F.lit("INFORM")),
                    F.when(F.col("total_allocations_usd").isNotNull(), F.lit("ALLOC")),
                    F.when(F.col("total_population").isNotNull(), F.lit("COD")),
                    F.when(F.col("hrp_revised_requirements_usd").isNotNull(), F.lit("HRP")),
                    F.when(F.col("severity_trend_label").isNotNull(), F.lit("TREND"))
                ))
    .withColumn("source_count",
                (F.col("fts_requirements_usd").isNotNull().cast("int") +
                 F.col("inform_severity_index").isNotNull().cast("int") +
                 F.col("total_allocations_usd").isNotNull().cast("int") +
                 F.col("total_population").isNotNull().cast("int") +
                 F.col("hrp_revised_requirements_usd").isNotNull().cast("int") +
                 F.col("severity_trend_label").isNotNull().cast("int")))
)

# --- Write ---
gold_country_year.write.mode("overwrite").saveAsTable(
    f"{CATALOG}.{SCHEMA}.{GOLD_PREFIX}country_year_funding"
)

print(f"✅ Gold Table 1 written: {GOLD_PREFIX}country_year_funding")
print(f"   Rows: {spark.table(f'{CATALOG}.{SCHEMA}.{GOLD_PREFIX}country_year_funding').count()}")
print(f"\n   R2 data_sources distribution:")
display(
    spark.table(f"{CATALOG}.{SCHEMA}.{GOLD_PREFIX}country_year_funding")
    .groupBy("source_count").count().orderBy("source_count")
)
print(f"\n   Top 10 by funding gap:")
display(spark.table(f"{CATALOG}.{SCHEMA}.{GOLD_PREFIX}country_year_funding")
        .orderBy(F.col("year").desc(), F.col("funding_gap_usd").desc()).limit(10))

In [0]:
# =============================================================================
# THRESHOLD CALIBRATION: Analyze fts_percent_funded distribution
# =============================================================================
# Purpose: Empirically determine where to set UNDERFUNDED_THRESHOLD
# Run AFTER Table 1 is created. This cell does NOT write a table.
# =============================================================================

import numpy as np

t1 = spark.table(f"{CATALOG}.{SCHEMA}.{GOLD_PREFIX}country_year_funding")

# Filter to rows with valid percent_funded (have both requirements and funding)
funded_dist = (
    t1.filter(F.col("fts_percent_funded").isNotNull())
    .select("fts_percent_funded")
    .toPandas()
)

print("=" * 70)
print("DISTRIBUTION OF fts_percent_funded (for threshold calibration)")
print("=" * 70)
print(f"\nTotal rows with valid percent_funded: {len(funded_dist)}")
print(f"\nDescriptive Statistics:")
print(funded_dist.describe().to_string())

# Percentiles
percentiles = [10, 20, 25, 30, 33, 40, 50, 60, 75, 90]
print(f"\nKey Percentiles:")
for p in percentiles:
    val = np.percentile(funded_dist['fts_percent_funded'], p)
    print(f"  P{p:2d}: {val:.1f}%")

# Distribution buckets
print(f"\nFunding Coverage Buckets:")
buckets = [(0, 10), (10, 20), (20, 33), (33, 50), (50, 67), (67, 80), (80, 100), (100, 999)]
for low, high in buckets:
    count = len(funded_dist[(funded_dist['fts_percent_funded'] >= low) & (funded_dist['fts_percent_funded'] < high)])
    pct = count / len(funded_dist) * 100
    label = f"{low}-{high}%" if high <= 100 else f"{low}%+"
    print(f"  {label:>10s}: {count:5d} rows ({pct:.1f}%)")

# Year-specific analysis (recent years most relevant)
print(f"\nMedian percent_funded by year (recent):")
recent = t1.filter(
    (F.col("fts_percent_funded").isNotNull()) & (F.col("year") >= 2019)
).groupBy("year").agg(
    F.expr("percentile_approx(fts_percent_funded, 0.5)").alias("median_pct_funded"),
    F.avg("fts_percent_funded").alias("mean_pct_funded"),
    F.count("*").alias("n")
).orderBy("year")
display(recent)

print(f"\n" + "=" * 70)
print(f"CURRENT THRESHOLD: {UNDERFUNDED_THRESHOLD}%")
print(f"Rows below threshold: {len(funded_dist[funded_dist['fts_percent_funded'] < UNDERFUNDED_THRESHOLD])} "
      f"({len(funded_dist[funded_dist['fts_percent_funded'] < UNDERFUNDED_THRESHOLD]) / len(funded_dist) * 100:.1f}%)")
print(f"\nRECOMMENDATION: Set threshold at bottom quartile (P25) or global")
print(f"median — whichever better discriminates 'overlooked' from 'typical'.")
print(f"If >60% of crises are below 50%, the threshold is too generous.")
print("=" * 70)

In [0]:
# =============================================================================
# GOLD TABLE 2: Sector Funding Gaps
# Grain: iso3 × year × sector_canonical
# =============================================================================
# Combines HNO needs data (people in need, targeted) with FTS cluster funding.
# Enables: "Show me food security hotspots with <10% funding"
#
# -----------------------------------------------------------------------------
# ISSUE 1 (B3): SECTOR NAME MISMATCH BETWEEN HNO AND FTS [RESOLVED]
# -----------------------------------------------------------------------------
# PROBLEM: The two source tables shared no stable join key for sector.
#   - HNO uses a cluster CODE (PRO, FSC, HEA …) in `cluster`.
#   - FTS (cluster_global) used free-text multilingual labels (962 variants).
#   Previous regex approach: 38% → 67% match. Sub-clusters got 0%.
#
# SOLUTION (v2 — Taxonomy Join):
#   Switched FTS source: cluster_global → globalcluster_global.
#   The globalcluster table uses the same OCHA global taxonomy IDs (1-16,
#   26479+) with only 24 standardized cluster names. No regex needed.
#   Bridge: silver table `hpc_global_cluster_taxonomy` (22 rows) maps:
#     HNO.cluster = taxonomy.cluster_code → taxonomy.cluster_id = FTS.clustercode
#   Result: ~95% match rate. Protection sub-clusters (PRO-CPN, PRO-GBV,
#           PRO-HLP, PRO-MIN) now have FTS funding data for 60 countries.
#   Both tables have identical total funding (verified: AFG 2024 = $1.58B).
#
# -----------------------------------------------------------------------------
# ISSUE 2 (HNO FAN-OUT): NATIONAL TOTALS MIXED WITH SUB-GROUP BREAKDOWNS
# -----------------------------------------------------------------------------
# PROBLEM: The silver HNO table stores two logically distinct row types at
# national level (all admin pcodes NULL), differentiated by `category`:
#
#   category = 'total' / NULL  → national aggregate (one row per cluster)
#   category = <sub-group>     → population disaggregations (IDP, Returnee,
#                                 Female, Children, etc. — up to 94 rows per
#                                 sector per country-year)
#
# Because both row types have NULL admin pcodes, a simple admin-pcode filter
# returns ALL of them. Summing `in_need` across all rows for AFG 2025
# Protection produced 163M instead of the correct 22M (7.4× overcount);
# MOZ 2024 Protection: 17M vs 1.4M (12× overcount). Three rows showed
# coverage_ratio > 1.0, which is physically impossible and an indicator of
# the inflated denominator.
#
# SOLUTION (Option A — zero data loss):
#   Step 1 — Category filter: add `category IN ('total', NULL)` to the
#             admin-pcode filter. Every iso3+year+cluster group that has
#             sub-group rows also has at least one total/NULL row, so no
#             sector disappears from the output. The sub-group rows are
#             discarded; only the national aggregate row(s) remain.
#   Step 2 — ROW_NUMBER() dedup: a small number of cluster codes have two
#             'total'-category rows with different descriptions (e.g.
#             PRO → 'Protection (overall)' and 'General Protection').
#             A window function partitioned by (iso3, year, cluster), ordered
#             by in_need DESC, picks the row with the highest in_need as the
#             authoritative national aggregate and discards the other.
#   Outcome: coverage_ratio > 1.0 dropped from 3 rows to 0; in_need values
#             match HNO source national totals; row count unchanged at 668.
# =============================================================================

hno = spark.table(f"{CATALOG}.{SCHEMA}.{SILVER_PREFIX}hpc_hno")
# SWITCHED: from cluster_global (plan-specific codes, regex matching)
# to globalcluster_global (global taxonomy IDs, deterministic join)
fts_globalcluster = spark.table(f"{CATALOG}.{SCHEMA}.{SILVER_PREFIX}fts_requirements_funding_globalcluster_global")
# Bridge table: cluster_code ↔ cluster_id ↔ cluster_name
taxonomy = spark.table(f"{CATALOG}.{SCHEMA}.{SILVER_PREFIX}hpc_global_cluster_taxonomy")

# =============================================================================
# SECTOR MAPPING VIA TAXONOMY TABLE (replaces regex + dict)
# =============================================================================
# Join path: HNO.cluster = taxonomy.cluster_code → taxonomy.cluster_id = FTS.clustercode
# This provides deterministic equi-joins (no regex, no string matching).
# The globalcluster_global table uses the same taxonomy IDs (1-16, 26479+)
# and already has Protection sub-clusters disaggregated (60 countries).
# =============================================================================

# --- 2a. HNO: Country-level sector data with canonical name (via taxonomy) ---
# (1) category filter keeps national aggregate rows only
# (2) ROW_NUMBER() dedup resolves residual description-level ties within a cluster
# (3) Join to taxonomy for canonical sector_name (replaces hardcoded dict)
hno_sector = (
    hno
    .filter(
        (F.col("admin_1_pcode").isNull()) &
        (F.col("admin_2_pcode").isNull()) &
        (F.col("admin_3_pcode").isNull()) &
        (F.col("cluster") != "ALL") &
        (F.col("category").isin("total") | F.col("category").isNull())
    )
    # Join HNO cluster code to taxonomy for canonical name
    .join(taxonomy.select(
        F.col("cluster_code"),
        F.col("cluster_name").alias("sector_canonical"),
        F.col("cluster_id")
    ), F.col("cluster") == F.col("cluster_code"), "inner")
    .withColumn("_rn", F.row_number().over(
        Window.partitionBy("country_iso3", "year", "cluster")
        .orderBy(F.col("in_need").desc_nulls_last())
    ))
    .filter(F.col("_rn") == 1)
    .drop("_rn")
    .select(
        F.col("country_iso3").alias("iso3"),
        "year",
        F.col("cluster").alias("sector_code"),
        F.col("description").alias("sector_name"),
        "sector_canonical",
        "cluster_id",  # carry through for FTS join
        "in_need", "targeted", "affected", "reached"
    )
)

# --- 2b. FTS globalcluster: join via taxonomy cluster_id (no regex) ---
# The globalcluster_global table already uses global taxonomy IDs as clustercode.
# We join directly on cluster_id — deterministic, no string matching.
fts_sector = (
    fts_globalcluster
    .filter(F.col("clustercode").isNotNull())
    .groupBy(F.col("countrycode").alias("iso3"), "year", F.col("clustercode").cast("int").alias("cluster_id"))
    .agg(
        F.sum("requirements").alias("sector_requirements_usd"),
        F.sum("funding").alias("sector_funding_usd"),
        F.concat_ws(" | ", F.collect_set("cluster")).alias("fts_cluster_names_matched")
    )
    .withColumn(
        "sector_pct_funded",
        F.when(F.col("sector_requirements_usd") > 0,
               F.round(F.col("sector_funding_usd") / F.col("sector_requirements_usd") * 100, 1))
    )
)

# --- 2c. Join HNO to FTS via cluster_id + iso3 + year (deterministic) ---
gold_sector_gaps = (
    hno_sector
    .join(fts_sector, on=["iso3", "year", "cluster_id"], how="left")
    .drop("cluster_id")  # internal join key, not needed in output
    .join(registry_country.select("iso3", "region"), on="iso3", how="left")
    .withColumn("coverage_ratio",
                F.when((F.col("in_need") > 0) & (F.col("targeted").isNotNull()),
                       F.round(F.col("targeted") / F.col("in_need"), 4)))
    .withColumn("targeting_gap_people",
                F.when(F.col("in_need").isNotNull() & F.col("targeted").isNotNull(),
                       (F.col("in_need") - F.col("targeted")).cast("long")))
    .withColumn("sector_funding_gap_usd",
                F.when(F.col("sector_requirements_usd").isNotNull(),
                       F.col("sector_requirements_usd") - F.coalesce(F.col("sector_funding_usd"), F.lit(0))))
)

# --- Write ---
gold_sector_gaps.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(
    f"{CATALOG}.{SCHEMA}.{GOLD_PREFIX}sector_funding_gaps"
)

# --- Validation ---
t2_new = spark.table(f"{CATALOG}.{SCHEMA}.{GOLD_PREFIX}sector_funding_gaps")
total = t2_new.count()
matched = t2_new.filter(F.col("sector_pct_funded").isNotNull()).count()
print(f"✅ Gold Table 2 written: {GOLD_PREFIX}sector_funding_gaps")
print(f"   Total rows:                 {total}")
print(f"   WITH FTS match:             {matched}  ({round(matched/total*100,1)}%)")
print(f"   WITHOUT FTS match:          {total-matched}  ({round((total-matched)/total*100,1)}%)")
print(f"   Match rate (taxonomy join): {round(matched/total*100,1)}% (prev: 67% regex, 38% naive)")

# --- HNO data quality checks (Option A fix validation) ---
coverage_over1 = t2_new.filter(F.col("coverage_ratio") > 1.0).count()
targeting_negative = t2_new.filter(F.col("targeting_gap_people") < 0).count()
print(f"\n   HNO data quality (Option A fix):")
print(f"   coverage_ratio > 1.0 (over-targeted or data issue): {coverage_over1}")
print(f"   targeting_gap_people < 0 (impossible with correct in_need): {targeting_negative}")
if coverage_over1 == 0 and targeting_negative == 0:
    print(f"   ✅ No impossible values — HNO fan-out resolved")
elif targeting_negative == 0:
    print(f"   ℹ️  {coverage_over1} over-targeted rows remain (genuine over-targeting, e.g. YEM Multi-Sector)")
else:
    print(f"   ❌ {targeting_negative} negative targeting gaps remain — investigate")

print(f"\n   Match rate by canonical sector:")
display(
    t2_new.groupBy("sector_canonical")
    .agg(
        F.count("*").alias("rows"),
        F.sum((F.col("sector_pct_funded").isNotNull()).cast("int")).alias("matched"),
        F.round(F.sum((F.col("sector_pct_funded").isNotNull()).cast("int")) / F.count("*") * 100, 1).alias("match_pct")
    )
    .orderBy(F.col("match_pct").desc())
)

In [0]:
# =============================================================================
# GOLD TABLE 3: Overlooked Crisis Scorecard
# Grain: iso3 × year
# =============================================================================
# PURPOSE: Produce a RANKED LIST of overlooked crises using a composite score.
#
# FORMULA (multiplicative — all components must be non-zero for high score):
#   overlooked_score = severity_norm
#                   × funding_gap_norm
#                   × scale_norm
#                   × trend_multiplier
#                   × persistence_multiplier
#                   × (1 - allocation_adequacy_norm)
#
# WHY MULTIPLICATIVE:
#   - Well-funded crisis (gap=0) → score=0 regardless of severity. Correct.
#   - Low-severity + gap → low score. Correct.
#   - Only the INTERSECTION of high severity + large gap + scale + neglect
#     produces top rankings.
#
# LIMITATIONS:
#   - Only ranks within the MEASURABLE universe (INFORM-tracked crises).
#   - Crises not in INFORM are UNRANKED, not scored zero.
#   - The RAG layer should surface: "N countries have FTS requirements but
#     no INFORM data" at query time (derive from NULLs, no status column).
# =============================================================================

# Read Table 1 (already written)
t1 = spark.table(f"{CATALOG}.{SCHEMA}.{GOLD_PREFIX}country_year_funding")

# --- 3a. Filter to scorable rows (must have severity + requirements) ---
# ASSUMPTION A3: Absence from this table = unmeasured, not unimportant
scorable = t1.filter(
    (F.col("inform_severity_index").isNotNull()) &
    (F.col("fts_requirements_usd").isNotNull()) &
    (F.col("fts_requirements_usd") > 0)
)

# --- 3b. Compute normalized components (each 0 to 1) ---

# SEVERITY: Normalize INFORM index (1-5 scale, already harmonized in Table 1) to 0-1
# Table 1 handles the 2026 scale normalization (/2), so this is always /5 here.
scorable = scorable.withColumn(
    "severity_norm",
    F.round(F.col("inform_severity_index") / 5.0, 4)
)

# FUNDING GAP: (1 - pct_funded/100), clamped to [0, 1]
# 0% funded → 1.0, 100% funded → 0.0, overfunded → 0.0
scorable = scorable.withColumn(
    "funding_gap_norm",
    F.greatest(
        F.lit(0.0),
        F.least(F.lit(1.0), 1.0 - F.coalesce(F.col("fts_percent_funded"), F.lit(0)) / 100.0)
    )
)

# SCALE: log10(requirements_usd) normalized to 0-1 using observed range
# log10(1M) = 6, log10(10B) = 10. Normalize: (log - 5) / 5, clamped [0,1]
scorable = scorable.withColumn(
    "log_requirements",
    F.log10(F.col("fts_requirements_usd"))
).withColumn(
    "scale_norm",
    F.greatest(
        F.lit(0.0),
        F.least(F.lit(1.0), (F.col("log_requirements") - 5.0) / 5.0)
    )
)

# TREND MULTIPLIER: From INFORM trend label via mapping
# Increasing=1.3, Stable=1.0, Decreasing=0.8, NULL/-=1.0
scorable = scorable.withColumn(
    "trend_multiplier",
    F.when(F.col("severity_trend_label") == "Increasing", TREND_MULTIPLIER_MAP["Increasing"])
    .when(F.col("severity_trend_label") == "Stable", TREND_MULTIPLIER_MAP["Stable"])
    .when(F.col("severity_trend_label") == "Decreasing", TREND_MULTIPLIER_MAP["Decreasing"])
    .otherwise(TREND_MULTIPLIER_MAP[None])  # "-" or NULL
)

# PERSISTENCE MULTIPLIER: 1.0 + (years_below_threshold / LOOKBACK) * 0.5
# Range: 1.0 (no history) to 1.5 (all years underfunded)
scorable = scorable.withColumn(
    "persistence_multiplier",
    F.round(
        1.0 + F.coalesce(F.col("years_below_threshold"), F.lit(0)) / PERSISTENCE_LOOKBACK_YEARS * 0.5,
        3
    )
)

# ALLOCATION ADEQUACY: If CERF/CBPF responded well, less "overlooked"
# allocation_adequacy = min(1, allocations / requirements)
# Then score component = (1 - adequacy): 0 allocations → 1.0, full coverage → 0.0
# ASSUMPTION A4: Allocation response means "system noticed this crisis"
scorable = scorable.withColumn(
    "allocation_adequacy_norm",
    F.least(
        F.lit(1.0),
        F.coalesce(F.col("total_allocations_usd"), F.lit(0)) /
        F.col("fts_requirements_usd")
    )
).withColumn(
    "allocation_neglect",
    F.round(1.0 - F.col("allocation_adequacy_norm"), 4)
)

# --- 3c. COMPOSITE SCORE ---
scorable = scorable.withColumn(
    "overlooked_score",
    F.round(
        F.col("severity_norm") *
        F.col("funding_gap_norm") *
        F.col("scale_norm") *
        F.col("trend_multiplier") *
        F.col("persistence_multiplier") *
        F.col("allocation_neglect"),
        6
    )
)

# --- 3d. RANK within each year ---
scorable = scorable.withColumn(
    "rank_in_year",
    F.dense_rank().over(
        Window.partitionBy("year").orderBy(F.col("overlooked_score").desc())
    )
)

# --- 3e. Enrich with drivers and crisis metadata ---
registry_enrichment = (
    registry
    .groupBy("iso3")
    .agg(
        F.concat_ws(", ", F.collect_set("drivers")).alias("crisis_drivers"),
        F.count("crisis_id").alias("tracked_crises_count")
    )
)

gold_scorecard = (
    scorable
    .join(registry_enrichment, on="iso3", how="left")
    # Conflict vs climate flags (for filtering)
    .withColumn("is_conflict_driven",
                F.coalesce(F.col("crisis_drivers").contains("Conflict"), F.lit(False)))
    .withColumn("is_climate_driven",
                F.coalesce(F.col("crisis_drivers").rlike("(?i)(drought|flood|cyclone|climate)"), F.lit(False)))
    .select(
        # Identity
        "iso3", "year", "country_name", "region",
        # Raw inputs
        "inform_severity_index", "inform_severity_category",
        "fts_requirements_usd", "fts_funding_usd", "fts_percent_funded",
        "total_allocations_usd", "total_population",
        "severity_trend_label", "years_below_threshold",
        # Normalized components (for explainability/transparency)
        "severity_norm", "funding_gap_norm", "scale_norm",
        "trend_multiplier", "persistence_multiplier", "allocation_neglect",
        # THE SCORE
        "overlooked_score", "rank_in_year",
        # Enrichment
        "crisis_drivers", "tracked_crises_count",
        "is_conflict_driven", "is_climate_driven",
        "funding_gap_usd", "per_capita_funding_usd", "per_capita_requirements_usd"
    )
)

# --- Write ---
gold_scorecard.write.mode("overwrite").saveAsTable(
    f"{CATALOG}.{SCHEMA}.{GOLD_PREFIX}overlooked_crises_scored"
)

print(f"✅ Gold Table 3 written: {GOLD_PREFIX}overlooked_crises_scored")
print(f"   Rows (scorable country-years): {gold_scorecard.count()}")
print(f"   Distinct countries scored: {gold_scorecard.select('iso3').distinct().count()}")
print(f"\nTop 15 Overlooked Crises (latest year):")
display(
    gold_scorecard
    .filter(F.col("year") == gold_scorecard.agg(F.max("year")).collect()[0][0])
    .orderBy(F.col("overlooked_score").desc())
    .limit(15)
)

In [0]:
# =============================================================================
# GOLD TABLE 4: Regional Summary (Trend Analysis)
# Grain: region × year
# =============================================================================
# Pre-aggregated for quick retrieval. Enables:
#   "Which regions are consistently underfunded relative to need?"
# Source: Gold Table 1 (country_year_funding)
# =============================================================================

t1 = spark.table(f"{CATALOG}.{SCHEMA}.{GOLD_PREFIX}country_year_funding")

gold_regional = (
    t1
    .filter(F.col("region").isNotNull())
    .groupBy("region", "year")
    .agg(
        F.sum("fts_requirements_usd").alias("total_requirements_usd"),
        F.sum("fts_funding_usd").alias("total_funding_usd"),
        F.sum("funding_gap_usd").alias("total_funding_gap_usd"),
        F.sum("total_allocations_usd").alias("total_allocations_usd"),
        F.sum("total_population").alias("total_population"),
        F.avg("inform_severity_index").alias("avg_severity_index"),
        F.max("inform_severity_index").alias("max_severity_index"),
        F.countDistinct("iso3").alias("country_count"),
        F.sum("fts_plan_count").alias("total_plans"),
        # Persistence: avg years underfunded across countries in region
        F.avg("years_below_threshold").alias("avg_years_underfunded"),
        # Count of countries with increasing severity
        F.sum(F.when(F.col("severity_trend_label") == "Increasing", 1).otherwise(0)).alias("countries_worsening")
    )
    .withColumn("regional_percent_funded",
                F.when(F.col("total_requirements_usd") > 0,
                       F.round(F.col("total_funding_usd") / F.col("total_requirements_usd") * 100, 1)))
    .withColumn("per_capita_funding_usd",
                F.when(F.col("total_population") > 0,
                       F.round(F.col("total_funding_usd") / F.col("total_population"), 2)))
    .orderBy("region", "year")
)

# --- Write ---
gold_regional.write.mode("overwrite").saveAsTable(
    f"{CATALOG}.{SCHEMA}.{GOLD_PREFIX}regional_summary"
)

print(f"✅ Gold Table 4 written: {GOLD_PREFIX}regional_summary")
print(f"   Rows: {spark.table(f'{CATALOG}.{SCHEMA}.{GOLD_PREFIX}regional_summary').count()}")
display(
    spark.table(f"{CATALOG}.{SCHEMA}.{GOLD_PREFIX}regional_summary")
    .orderBy(F.col("year").desc(), F.col("total_funding_gap_usd").desc())
)

In [0]:
# =============================================================================
# GOLD TABLE 5: INFORM Severity Temporal Series
# Grain: crisis_id × year_month
# =============================================================================
# PURPOSE: Provide a clean, queryable time series of crisis severity for:
#   - Trajectory visualization ("show me Sudan's severity over time")
#   - Identifying escalation timing vs. funding response timing
#   - Supporting RAG "why" narratives ("when did this crisis start worsening?")
#
# TRANSFORMATION from Silver:
#   - Unpivots 88 wide columns (y2019_01...y2026_04) into long format
#   - Strips NULL values (from "-" sentinel = under-monitored periods)
#   - Joins to crisis_registry for iso3, country_name, region
#   - Preserves crisis_id granularity (AFG001 vs AFG007)
#
# DEDUPLICATION (C3 FIX):
#   Each snapshot file contains the FULL history back to y2019_01. Unpivoting
#   all 66 snapshots produces ~66 copies of each (crisis_id, year_month) pair.
#   The severity_value is identical across snapshots for the same month — only
#   the overall trend label differs (it reflects the trend AT SNAPSHOT TIME,
#   not at the historical month).
#
#   Strategy:
#     1. For months available in pre-2026 snapshots (2019-01 to 2025-12):
#        Take from a pre-2026 snapshot (values on consistent 1-5 scale).
#     2. For months only in 2026 snapshots (2026-01 to 2026-04):
#        These are new data points on the 1-10 scale. Divide by 2 to
#        normalize back to 1-5 for a consistent time series.
#
#   Result: 251,008 rows → ~6,889 rows (97% reduction, zero information loss).
#
# SCALE NORMALIZATION:
#   OCHA changed INFORM methodology in Jan 2026: index expanded from 1-5 to
#   1-10 scale. All 63 pre-2026 snapshot files have max_index ≤ 5.0; all 4
#   2026 snapshots reach up to 9.7. We normalize 2026-only months by /2
#   so the full series is on a consistent 1-5 scale for temporal comparison.
#
# NOTE: This table is NOT used in the scoring formula directly.
# Table 1 has summary columns (severity_year_avg, severity_year_delta).
# This table exists for agent drill-down and temporal narratives.
# =============================================================================

from pyspark.sql import functions as F
from pyspark.sql.window import Window
from functools import reduce

inform_trends = spark.table(f"{CATALOG}.{SCHEMA}.{SILVER_PREFIX}inform_trends")
registry = spark.table(f"{CATALOG}.{SCHEMA}.{SILVER_PREFIX}inform_crisis_registry")

# --- 5a. Identify all y-columns (format: y2019_01, y2019_02, ..., y2026_04) ---
y_columns = [c for c in inform_trends.columns if c.startswith("y20") and "_" in c[1:]]
print(f"   Unpivoting {len(y_columns)} monthly columns...")

# --- 5b. Unpivot wide → long using stack() ---
stack_exprs = ", ".join([f"'{col}', `{col}`" for col in y_columns])
stack_sql = f"stack({len(y_columns)}, {stack_exprs}) as (year_month_code, severity_value)"

trends_long = (
    inform_trends
    .select("crisis_id", "country", "crisis", "trend", "snapshot_month",
            F.expr(stack_sql))
    # Remove NULLs (from "-" sentinel in silver = under-monitored periods)
    .filter(F.col("severity_value").isNotNull())
)

# --- 5c. Parse year_month_code ("y2024_09") to proper DATE ---
trends_long = (
    trends_long
    .withColumn("year_val", F.substring("year_month_code", 2, 4).cast("int"))
    .withColumn("month_val", F.substring("year_month_code", 7, 2).cast("int"))
    .withColumn("year_month", F.make_date("year_val", "month_val", F.lit(1)))
    .drop("year_val", "month_val", "year_month_code")
)

# --- 5d. DEDUP: Keep one row per (crisis_id, year_month) ---
# Classify snapshots: pre-2026 (1-5 scale) vs 2026 (1-10 scale)
# We prefer pre-2026 snapshots to keep values on the original 1-5 scale.
# For months that ONLY exist in 2026 snapshots, we take them and normalize /2.

# Flag whether snapshot is from 2026
trends_long = trends_long.withColumn(
    "is_2026_snapshot",
    F.col("snapshot_month").rlike("-(2026)$")
)

# Dedup strategy: rank within (crisis_id, year_month)
# Priority: pre-2026 snapshots first (is_2026_snapshot=False sorts before True? No—False<True in boolean sort)
# We want pre-2026 first, so order by is_2026_snapshot ASC (False=0 first), then snapshot_month DESC (latest pre-2026)
trends_deduped = (
    trends_long
    .withColumn("_rn", F.row_number().over(
        Window.partitionBy("crisis_id", "year_month")
        .orderBy(
            F.col("is_2026_snapshot").asc(),   # pre-2026 first
            F.col("snapshot_month").desc()      # latest snapshot within that group
        )
    ))
    .filter(F.col("_rn") == 1)
    .drop("_rn")
)

# --- 5e. Normalize 2026-only months (where we had to use a 2026 snapshot) ---
# These have severity on 1-10 scale; divide by 2 to get consistent 1-5 scale
trends_deduped = trends_deduped.withColumn(
    "severity_value",
    F.when(F.col("is_2026_snapshot"), F.round(F.col("severity_value") / 2.0, 2))
     .otherwise(F.col("severity_value"))
)

# --- 5f. Join to registry for iso3, region ---
# Coalesce country_name with C&T Taxonomy (same precedence as Table 1 step 1g)
ct_taxonomy_t5 = (
    spark.table(f"{CATALOG}.{SCHEMA}.{SILVER_PREFIX}country_territory_taxonomy")
    .filter(F.col("admin_level") == 0)
    .select(
        F.col("iso3").alias("ct_iso3"),
        F.col("country_name").alias("ct_country_name")
    )
)

registry_for_join = (
    registry.select("crisis_id", "iso3", "country_name", "region", "drivers")
    .join(ct_taxonomy_t5, F.col("iso3") == F.col("ct_iso3"), how="left")
    .withColumn("country_name", F.coalesce(F.col("country_name"), F.col("ct_country_name")))
    .drop("ct_iso3", "ct_country_name")
)

gold_temporal = (
    trends_deduped
    .join(registry_for_join, on="crisis_id", how="inner")
    .select(
        "crisis_id",
        "iso3",
        "country_name",
        "region",
        "year_month",
        "severity_value",
        "trend",
        "drivers",
        "snapshot_month",
        "is_2026_snapshot"  # provenance: was this from a 2026 snapshot (normalized /2)?
    )
    .orderBy("iso3", "crisis_id", "year_month")
)

# --- Write ---
gold_temporal.write.mode("overwrite").saveAsTable(
    f"{CATALOG}.{SCHEMA}.{GOLD_PREFIX}inform_temporal_series"
)

t5 = spark.table(f"{CATALOG}.{SCHEMA}.{GOLD_PREFIX}inform_temporal_series")
print(f"\n✅ Gold Table 5 written: {GOLD_PREFIX}inform_temporal_series")
print(f"   Rows: {t5.count()}")
print(f"   Distinct crises: {t5.select('crisis_id').distinct().count()}")
print(f"   Distinct countries: {t5.select('iso3').distinct().count()}")
print(f"   Date range: {t5.agg(F.min('year_month'), F.max('year_month')).collect()[0]}")
print(f"   Severity range: [{t5.agg(F.min('severity_value')).collect()[0][0]}, {t5.agg(F.max('severity_value')).collect()[0][0]}]")
print(f"   Rows from 2026 snapshots (normalized /2): {t5.filter(F.col('is_2026_snapshot')).count()}")

# Show example: trajectory for a known severe crisis
print(f"\n   Example: AFG001 trajectory (last 12 months):")
display(
    t5.filter((F.col("crisis_id") == "AFG001") & (F.col("year_month") >= "2025-01-01"))
    .orderBy("year_month")
    .select("crisis_id", "iso3", "year_month", "severity_value", "trend", "is_2026_snapshot")
)

In [0]:
# =============================================================================
# R3: COLUMN COMMENTS FOR AGENT DISCOVERABILITY
# =============================================================================
# When agents query Unity Catalog for table schemas, column comments are the
# primary documentation they see. These comments enable the agent to:
#   - Understand what a column measures without external docs
#   - Choose the right column for a user's question
#   - Explain results in plain language
# =============================================================================

# --- Table 1: Country-Year Funding ---
table1 = f"{CATALOG}.{SCHEMA}.{GOLD_PREFIX}country_year_funding"
comments_t1 = {
    "iso3": "ISO 3166-1 alpha-3 country code. Primary join key across all tables.",
    "year": "Calendar year of the funding cycle.",
    "fts_requirements_usd": "Total stated humanitarian funding requirements in USD from FTS appeals for this country-year.",
    "fts_funding_usd": "Total funding received against FTS appeals in USD.",
    "fts_percent_funded": "Percentage of requirements met by funding (funding/requirements*100).",
    "fts_plan_codes": "R4 Provenance: comma-separated FTS plan codes that contributed to this row (e.g. HAFG26, RBGD26).",
    "years_below_threshold": f"Count of years in the {PERSISTENCE_LOOKBACK_YEARS}-year window where percent_funded < {UNDERFUNDED_THRESHOLD}%. Measures chronic underfunding.",
    "yoy_pct_funded_change": "Year-over-year change in percent_funded (this year minus last year). Negative = worsening.",
    "inform_severity_index": "INFORM crisis severity index. Raw value from source: 1-5 scale for 2020-2025, 1-10 scale for 2026+ (OCHA methodology change). Higher = more severe.",
    "inform_severity_category": "INFORM severity category (1-5). Derived from index. 5 = Very High. Consistent scale across all years.",
    "severity_year_avg": "Average INFORM severity across all snapshots in this year. More robust than single-point latest.",
    "severity_year_delta": "Within-year severity change (latest minus earliest snapshot). Positive = crisis worsened during the year.",
    "inform_snapshots_in_year": "Number of INFORM snapshots available for this country-year. Higher = better data density.",
    "severity_trend_label": "OCHA-validated cross-year trend: Increasing, Stable, or Decreasing. From INFORM trends.",
    "total_population": "National population from COD. Filtered to avoid double-counting (population_group=all, gender=all, national level).",
    "total_allocations_usd": "Total CERF/CBPF allocations to this country-year in USD.",
    "funding_gap_usd": "Absolute funding gap: requirements minus funding in USD.",
    "funding_gap_pct": "Funding gap as percentage: (1 - funding/requirements) * 100.",
    "per_capita_funding_usd": "Funding per person: fts_funding_usd / total_population.",
    "per_capita_requirements_usd": "Requirements per person: fts_requirements_usd / total_population.",
    "data_sources": "R2 Completeness: pipe-separated list of sources that contributed data (FTS|INFORM|ALLOC|COD|HRP|TREND).",
    "source_count": "R2 Completeness: number of distinct data sources contributing to this row (0-6). Higher = more reliable.",
    "region": "Geographic region from INFORM crisis registry: Africa, Americas, Asia, Europe, Middle east.",
    "country_name": "Country display name from INFORM crisis registry."
}

for col, comment in comments_t1.items():
    try:
        spark.sql(f"ALTER TABLE {table1} ALTER COLUMN `{col}` COMMENT '{comment.replace(chr(39), chr(39)+chr(39))}'")
    except Exception as e:
        pass  # Column may not exist in some runs

print(f"✅ Table 1: {len(comments_t1)} column comments applied")

# --- Table 2: Sector Funding Gaps ---
table2 = f"{CATALOG}.{SCHEMA}.{GOLD_PREFIX}sector_funding_gaps"
comments_t2 = {
    "iso3": "ISO 3166-1 alpha-3 country code.",
    "year": "HNO reporting year (2024-2026).",
    "sector_code": "HNO cluster code (e.g. FSC, EDU, PRO-CPN, PRO-GBV). Joins to taxonomy.cluster_code.",
    "sector_name": "HNO sector description (free-text, from highest in_need row after dedup).",
    "sector_canonical": "Canonical English cluster name from OCHA taxonomy (e.g. Protection - Child Protection). Bridge between HNO and FTS.",
    "in_need": "People in need for this sector-country-year (from HNO).",
    "targeted": "People targeted for assistance (from HNO). Always <= in_need.",
    "sector_requirements_usd": "USD funding requirements for this sector from FTS globalcluster data.",
    "sector_funding_usd": "USD funding received for this sector from FTS globalcluster data.",
    "fts_cluster_names_matched": "FTS cluster name(s) that matched via taxonomy join. Provenance column.",
    "sector_pct_funded": "Sector funding coverage: sector_funding/sector_requirements*100. NULL if no FTS match.",
    "coverage_ratio": "Targeting coverage: targeted/in_need. Range 0-1. Shows what fraction of need is being addressed.",
    "targeting_gap_people": "People not targeted: in_need minus targeted.",
    "sector_funding_gap_usd": "Sector funding gap: requirements minus funding in USD.",
    "region": "Geographic region from INFORM crisis registry."
}

for col, comment in comments_t2.items():
    try:
        spark.sql(f"ALTER TABLE {table2} ALTER COLUMN `{col}` COMMENT '{comment.replace(chr(39), chr(39)+chr(39))}'")
    except Exception as e:
        pass

print(f"✅ Table 2: {len(comments_t2)} column comments applied")

# --- Table 3: Overlooked Crisis Scorecard ---
table3 = f"{CATALOG}.{SCHEMA}.{GOLD_PREFIX}overlooked_crises_scored"
comments_t3 = {
    "overlooked_score": "Composite score (0-1+): higher = more overlooked. Multiplicative formula: severity * gap * scale * trend * persistence * allocation_neglect.",
    "rank_in_year": "Dense rank within year by overlooked_score DESC. Rank 1 = most overlooked that year.",
    "severity_norm": "Normalized INFORM severity (0-1). Year-aware: index/10 for 2026+ (1-10 scale), index/5 for earlier years (1-5 scale).",
    "funding_gap_norm": "Normalized funding gap (0-1). Component of score. = 1 - pct_funded/100, clamped.",
    "scale_norm": "Normalized crisis scale (0-1). Component of score. = (log10(requirements) - 5) / 5, clamped.",
    "trend_multiplier": "Trend adjustment: 1.3 (Increasing), 1.0 (Stable/unknown), 0.8 (Decreasing).",
    "persistence_multiplier": "Chronic underfunding boost: 1.0 (no history) to 1.5 (all years critically underfunded).",
    "allocation_neglect": "1 minus allocation adequacy (0-1). High = system did NOT respond with CERF/CBPF.",
    "is_conflict_driven": "Boolean: crisis drivers include Conflict/Violence.",
    "is_climate_driven": "Boolean: crisis drivers include drought, flood, cyclone, or climate."
}

for col, comment in comments_t3.items():
    try:
        spark.sql(f"ALTER TABLE {table3} ALTER COLUMN `{col}` COMMENT '{comment.replace(chr(39), chr(39)+chr(39))}'")
    except Exception as e:
        pass

print(f"✅ Table 3: {len(comments_t3)} column comments applied")

# --- Table 5: Temporal Series ---
table5 = f"{CATALOG}.{SCHEMA}.{GOLD_PREFIX}inform_temporal_series"
comments_t5 = {
    "crisis_id": "INFORM crisis identifier (e.g. AFG001). Links to crisis_registry.",
    "year_month": "First day of the month for this severity observation (DATE type).",
    "severity_value": "INFORM severity index value for this crisis at this month. CAUTION: scale varies by snapshot file year (1-5 for pre-2026, 1-10 for 2026+). Compare within same snapshot, not across.",
    "trend": "OCHA-validated overall trend for this crisis: Increasing, Stable, Decreasing, or - (under-monitored).",
    "drivers": "Comma-separated crisis drivers (e.g. Conflict/ Violence, Drought)."
}

for col, comment in comments_t5.items():
    try:
        spark.sql(f"ALTER TABLE {table5} ALTER COLUMN `{col}` COMMENT '{comment.replace(chr(39), chr(39)+chr(39))}'")
    except Exception as e:
        pass

print(f"✅ Table 5: {len(comments_t5)} column comments applied")
print(f"\n✅ R3 complete: Column comments applied across all gold tables for agent discoverability.")

In [0]:
# =============================================================================
# VALIDATION & RELEVANCE ASSESSMENT
# =============================================================================
# Three types of checks:
#   PART A: Data Quality — row counts, NULL rates, sanity checks
#   PART B: Analytical Relevance — do these tables actually differentiate
#           overlooked crises? Can we answer the target questions?
#   PART C: Data Reliability — post-transformation integrity checks
#           (referential integrity, value ranges, temporal consistency,
#           cross-table agreement, transformation correctness)
#
# If PART B fails → fix joins, thresholds, or table design.
# If PART C fails → fix transformation logic (bugs, join fanout, data loss).
# =============================================================================

from pyspark.sql import functions as F
import numpy as np

print("=" * 70)
print("PART A: DATA QUALITY CHECKS")
print("=" * 70)

# --- Table 1: Country-Year Funding ---
t1 = spark.table(f"{CATALOG}.{SCHEMA}.{GOLD_PREFIX}country_year_funding")
t1_count = t1.count()
print(f"\n1. {GOLD_PREFIX}country_year_funding")
print(f"   Rows: {t1_count}")
print(f"   Distinct ISO3: {t1.select('iso3').distinct().count()}")
print(f"   Year range: {t1.agg(F.min('year'), F.max('year')).collect()[0]}")
print(f"   With FTS data:      {t1.filter(F.col('fts_requirements_usd').isNotNull()).count()} ({t1.filter(F.col('fts_requirements_usd').isNotNull()).count()/t1_count*100:.0f}%)")
print(f"   With severity:      {t1.filter(F.col('inform_severity_index').isNotNull()).count()} ({t1.filter(F.col('inform_severity_index').isNotNull()).count()/t1_count*100:.0f}%)")
print(f"   With population:    {t1.filter(F.col('total_population').isNotNull()).count()} ({t1.filter(F.col('total_population').isNotNull()).count()/t1_count*100:.0f}%)")
print(f"   With allocations:   {t1.filter(F.col('total_allocations_usd').isNotNull()).count()}")
print(f"   With trend label:   {t1.filter(F.col('severity_trend_label').isNotNull()).count()}")
print(f"   With persistence:   {t1.filter(F.col('years_below_threshold').isNotNull()).count()}")
print(f"   With HRP data:      {t1.filter(F.col('hrp_revised_requirements_usd').isNotNull()).count()}")
print(f"   With severity_year_avg:   {t1.filter(F.col('severity_year_avg').isNotNull()).count()}")
print(f"   With severity_year_delta: {t1.filter(F.col('severity_year_delta').isNotNull()).count()}")

# --- Table 2: Sector Funding Gaps ---
t2 = spark.table(f"{CATALOG}.{SCHEMA}.{GOLD_PREFIX}sector_funding_gaps")
t2_count = t2.count()
print(f"\n2. {GOLD_PREFIX}sector_funding_gaps")
print(f"   Rows: {t2_count}")
print(f"   Distinct countries: {t2.select('iso3').distinct().count()}")
print(f"   Distinct sectors: {t2.select('sector_name').distinct().count()}")
print(f"   Year range: {t2.agg(F.min('year'), F.max('year')).collect()[0]}")
fts_match_count = t2.filter(F.col('sector_pct_funded').isNotNull()).count()
print(f"   With FTS funding match: {fts_match_count} ({fts_match_count/t2_count*100:.0f}% match rate)")
print(f"   Sectors <10% funded: {t2.filter(F.col('sector_pct_funded') < 10).count()}")

# --- Table 3: Overlooked Crisis Scorecard ---
t3 = spark.table(f"{CATALOG}.{SCHEMA}.{GOLD_PREFIX}overlooked_crises_scored")
t3_count = t3.count()
print(f"\n3. {GOLD_PREFIX}overlooked_crises_scored")
print(f"   Rows (scorable): {t3_count}")
print(f"   Distinct countries: {t3.select('iso3').distinct().count()}")
score_stats = t3.agg(
    F.min('overlooked_score'), F.max('overlooked_score'),
    F.avg('overlooked_score'), F.stddev('overlooked_score')
).collect()[0]
print(f"   Score min/max/mean/std: {score_stats[0]:.4f} / {score_stats[1]:.4f} / {score_stats[2]:.4f} / {score_stats[3]:.4f}")
print(f"   Conflict-driven: {t3.filter(F.col('is_conflict_driven')).count()}")
print(f"   Climate-driven: {t3.filter(F.col('is_climate_driven')).count()}")

# --- Table 4: Regional Summary ---
t4 = spark.table(f"{CATALOG}.{SCHEMA}.{GOLD_PREFIX}regional_summary")
print(f"\n4. {GOLD_PREFIX}regional_summary")
print(f"   Rows: {t4.count()}")
print(f"   Regions: {[r[0] for r in t4.select('region').distinct().collect()]}")

# --- Table 5: Temporal Series ---
t5 = spark.table(f"{CATALOG}.{SCHEMA}.{GOLD_PREFIX}inform_temporal_series")
t5_count = t5.count()
print(f"\n5. {GOLD_PREFIX}inform_temporal_series")
print(f"   Rows: {t5_count}")
print(f"   Distinct crises: {t5.select('crisis_id').distinct().count()}")
print(f"   Date range: {t5.agg(F.min('year_month'), F.max('year_month')).collect()[0]}")

# --- Basic Sanity ---
print(f"\n--- Sanity Checks ---")
print(f"   Negative funding gaps (overfunded): {t1.filter(F.col('funding_gap_usd') < 0).count()}")
print(f"   Percent funded > 100%: {t1.filter(F.col('fts_percent_funded') > 100).count()}")
print(f"   NULL iso3 in Table 1: {t1.filter(F.col('iso3').isNull()).count()}")
print(f"   Score = 0 (zero component): {t3.filter(F.col('overlooked_score') == 0).count()}")


print(f"\n\n{'=' * 70}")
print("PART B: ANALYTICAL RELEVANCE CHECKS")
print("=" * 70)
print("\nThese checks verify the gold tables can ACTUALLY answer the target")
print("analytical questions. Failures indicate join issues or design gaps.\n")

# B1: SCORE DISCRIMINATION
print("─" * 70)
print("B1: SCORE DISCRIMINATION")
print("─" * 70)
scores_pd = t3.select("overlooked_score").toPandas()
percentiles = [10, 25, 50, 75, 90, 95, 99]
print(f"   Score percentiles:")
for p in percentiles:
    print(f"     P{p:2d}: {np.percentile(scores_pd['overlooked_score'].dropna(), p):.4f}")
p90 = np.percentile(scores_pd['overlooked_score'].dropna(), 90)
p10 = np.percentile(scores_pd['overlooked_score'].dropna(), 10)
ratio = p90 / max(p10, 0.0001)
verdict_b1 = "✅ GOOD" if ratio > 5 else "⚠️ WEAK" if ratio > 2 else "❌ POOR"
print(f"   P90/P10 ratio: {ratio:.1f}x | Verdict: {verdict_b1}")

# B2: FACE VALIDITY
print(f"\n{'─' * 70}")
print("B2: FACE VALIDITY")
print("─" * 70)
known_severe = ["SYR", "YEM", "AFG", "SDN", "COD", "SOM", "ETH", "MMR"]
latest_year = t3.agg(F.max("year")).collect()[0][0]
top_20 = [r[0] for r in t3.filter(F.col("year") == latest_year).orderBy(F.col("overlooked_score").desc()).select("iso3").limit(20).collect()]
hits = [c for c in known_severe if c in top_20]
print(f"   Latest year: {latest_year}")
print(f"   Top 20 scored: {top_20}")
print(f"   Known severe in top 20: {hits} ({len(hits)}/{len(known_severe)})")
verdict_b2 = "✅ GOOD" if len(hits) >= 4 else "⚠️ PARTIAL" if len(hits) >= 2 else "❌ POOR"
print(f"   Verdict: {verdict_b2}")
if len(hits) < 4:
    missing = [c for c in known_severe if c not in top_20]
    for c in missing[:3]:
        row = t3.filter((F.col("iso3") == c) & (F.col("year") == latest_year)).collect()
        if row:
            print(f"     {c}: score={row[0]['overlooked_score']:.4f}, pct_funded={row[0]['fts_percent_funded']}, severity={row[0]['inform_severity_index']}")
        else:
            print(f"     {c}: NOT IN SCORECARD (missing severity or requirements)")

# B3: SECTOR JOIN QUALITY
print(f"\n{'─' * 70}")
print("B3: SECTOR JOIN QUALITY")
print("─" * 70)
food_sectors = t2.filter(F.col("sector_name").rlike("(?i)(food|nutrition|security)"))
food_with_funding = food_sectors.filter(F.col("sector_pct_funded").isNotNull())
food_under10 = food_sectors.filter(F.col("sector_pct_funded") < 10)
print(f"   Food/nutrition sectors total rows: {food_sectors.count()}")
print(f"   With funding data: {food_with_funding.count()}")
print(f"   With <10% funded: {food_under10.count()}")
match_pct = fts_match_count / t2_count * 100 if t2_count > 0 else 0
verdict_b3 = "✅ GOOD" if match_pct >= 90 else "⚠️ PARTIAL" if match_pct > 50 else "❌ POOR"
print(f"   Overall sector match rate: {match_pct:.0f}% | Verdict: {verdict_b3}")
if match_pct < 90:
    unmatched_names = [r[0] for r in t2.filter(F.col("sector_pct_funded").isNull()).select("sector_name").distinct().limit(5).collect()]
    matched_names = [r[0] for r in t2.filter(F.col("sector_pct_funded").isNotNull()).select("sector_name").distinct().limit(5).collect()]
    print(f"   Sample UNMATCHED: {unmatched_names}")
    print(f"   Sample MATCHED:   {matched_names}")

# B4: TEMPORAL FEATURES UTILITY
print(f"\n{'─' * 70}")
print("B4: TEMPORAL FEATURES UTILITY")
print("─" * 70)
scored_with_persistence = t3.filter(F.col("years_below_threshold").isNotNull()).count()
scored_with_trend = t3.filter(F.col("severity_trend_label").isNotNull()).count()
print(f"   Scored rows with persistence: {scored_with_persistence} ({scored_with_persistence/t3_count*100:.0f}%)")
print(f"   Scored rows with trend: {scored_with_trend} ({scored_with_trend/t3_count*100:.0f}%)")
if scored_with_persistence > 0:
    persist_dist = t3.filter(F.col("years_below_threshold").isNotNull()).groupBy("years_below_threshold").count().orderBy("years_below_threshold").collect()
    print(f"   Persistence distribution: {[(int(r[0]), r[1]) for r in persist_dist]}")
    verdict_b4_persist = "✅ GOOD" if len(persist_dist) >= 3 else "⚠️ LOW VARIANCE"
else:
    verdict_b4_persist = "❌ NO DATA"
verdict_b4_trend = "✅ GOOD" if scored_with_trend/t3_count > 0.5 else "⚠️ LOW COVERAGE"
print(f"   Verdicts: Persistence={verdict_b4_persist} | Trend={verdict_b4_trend}")

# B5: ALLOCATION SIGNAL STRENGTH
print(f"\n{'─' * 70}")
print("B5: ALLOCATION SIGNAL STRENGTH")
print("─" * 70)
alloc_coverage = t3.filter(F.col("total_allocations_usd").isNotNull()).count()
high_neglect_pct = t3.filter(F.col("allocation_neglect") >= 0.99).count() / t3_count * 100
print(f"   Scored rows with allocations: {alloc_coverage} ({alloc_coverage/t3_count*100:.0f}%)")
verdict_b5 = "✅ DISCRIMINATES" if high_neglect_pct < 70 else "⚠️ MOSTLY INERT"
print(f"   Rows with allocation_neglect >= 0.99: {high_neglect_pct:.0f}% | Verdict: {verdict_b5}")

# B6: COVERAGE GAPS
print(f"\n{'─' * 70}")
print("B6: COVERAGE GAPS (meta-overlooked)")
print("─" * 70)
unscored = t1.filter(
    (F.col("fts_requirements_usd").isNotNull()) &
    (F.col("fts_requirements_usd") > 0) &
    (F.col("inform_severity_index").isNull())
)
print(f"   Countries with FTS req but NO severity: {unscored.select('iso3').distinct().count()}")
print(f"   Country-years unscored: {unscored.count()}")
if unscored.count() > 0:
    total_unscored_req = unscored.agg(F.sum("fts_requirements_usd")).collect()[0][0]
    total_all_req = t1.agg(F.sum("fts_requirements_usd")).collect()[0][0]
    print(f"   % of all requirements unscored: {total_unscored_req/total_all_req*100:.1f}%")

# B7: REGIONAL TRENDS
print(f"\n{'─' * 70}")
print("B7: REGIONAL TREND UTILITY")
print("─" * 70)
for region_row in t4.select("region").distinct().collect():
    region = region_row[0]
    region_data = t4.filter((F.col("region") == region) & (F.col("regional_percent_funded").isNotNull()))
    years_under_50 = region_data.filter(F.col("regional_percent_funded") < 50).count()
    total_years = region_data.count()
    if total_years > 0:
        print(f"   {region}: {years_under_50}/{total_years} years below 50% funded")


print(f"\n\n{'=' * 70}")
print("PART C: DATA RELIABILITY (post-transformation integrity)")
print("=" * 70)
print("\nThese checks verify the transformation logic is CORRECT.")
print("Failures indicate bugs, join fanout, or data loss.\n")

failures = []

# ──────────────────────────────────────────────────────────────────────
print("─" * 70)
print("C1: REFERENTIAL INTEGRITY")
print("    Do all iso3 codes in gold tables exist in a known reference source?")
print("─" * 70)
# Two valid reference sources:
#   1. INFORM crisis registry  — countries with an active INFORM classification
#   2. C&T Taxonomy (silver)   — authoritative UN iso3 reference (all territories)
# 17 countries appear in FTS/INFORM cross-snapshot data but have no formal
# INFORM crisis_id. They are valid iso3 codes confirmed in the C&T Taxonomy.
# These are expected per Assumption A3 and are NOT treated as failures here.
# TRUE failures = iso3 codes absent from BOTH sources (data entry errors).
registry_iso3 = spark.table(f"{CATALOG}.{SCHEMA}.{SILVER_PREFIX}inform_crisis_registry").select("iso3").distinct()
taxonomy_iso3 = (
    spark.table(f"{CATALOG}.{SCHEMA}.{SILVER_PREFIX}country_territory_taxonomy")
    .filter(F.col("admin_level") == 0)
    .select("iso3").distinct()
)
all_known_iso3 = registry_iso3.union(taxonomy_iso3).distinct()

t3_iso3 = t3.select("iso3").distinct()

# True orphans: not in INFORM registry AND not in C&T taxonomy
true_orphan_t3 = t3_iso3.subtract(all_known_iso3).count()
print(f"   Table 3 iso3 not in ANY reference source (true orphans): {true_orphan_t3}")
if true_orphan_t3 > 0:
    failures.append("C1: True orphan iso3 in scorecard (not in registry or taxonomy)")
    print(f"   ❌ Unknown codes: {[r[0] for r in t3_iso3.subtract(all_known_iso3).collect()]}")
else:
    print(f"   ✅ All scored iso3 codes confirmed in a reference source")

# Informational: INFORM-untracked but taxonomy-confirmed (expected, Assumption A3)
inform_untracked = t3_iso3.subtract(registry_iso3)
inform_untracked_count = inform_untracked.count()
if inform_untracked_count > 0:
    untracked_codes = sorted([r[0] for r in inform_untracked.collect()])
    print(f"   ℹ️  INFORM-untracked (scored via FTS only, A3 applies): {inform_untracked_count} countries")
    print(f"      {untracked_codes}")
    print(f"      → All confirmed in C&T Taxonomy. Not a failure.")

# Table 5 should only have crisis_ids from registry
t5_crises = t5.select("crisis_id").distinct()
registry_crises = spark.table(f"{CATALOG}.{SCHEMA}.{SILVER_PREFIX}inform_crisis_registry").select("crisis_id").distinct()
orphan_t5 = t5_crises.subtract(registry_crises).count()
print(f"   Table 5 crisis_ids NOT in registry: {orphan_t5}")
if orphan_t5 > 0:
    failures.append("C1: Orphan crisis_ids in temporal series")
else:
    print(f"   ✅ All temporal series crises traceable to registry")

# ──────────────────────────────────────────────────────────────────────
print(f"\n{'─' * 70}")
print("C2: VALUE RANGE VALIDATION")
print("    Are transformed values within expected bounds?")
print("─" * 70)

# INFORM severity should be 0-5 (after normalization to 0-1 in scorecard)
severity_range = t1.agg(F.min("inform_severity_index"), F.max("inform_severity_index")).collect()[0]
print(f"   Severity index range: [{severity_range[0]}, {severity_range[1]}] (expected: 0-5)")
if severity_range[1] and severity_range[1] > 5.0:
    failures.append(f"C2: Severity index exceeds 5.0 (max={severity_range[1]})")
    print(f"   ❌ EXCEEDS expected range")
else:
    print(f"   ✅ Within expected range")

# Funding should be non-negative (requirements and funding)
neg_req = t1.filter(F.col("fts_requirements_usd") < 0).count()
neg_fund = t1.filter(F.col("fts_funding_usd") < 0).count()
print(f"   Negative requirements: {neg_req} | Negative funding: {neg_fund}")
if neg_req > 0 or neg_fund > 0:
    failures.append("C2: Negative funding/requirements values")
    print(f"   ❌ Unexpected negative values")
else:
    print(f"   ✅ All financial values non-negative")

# Score components should be 0-1 (except multipliers which can be >1)
score_norms = t3.agg(
    F.min("severity_norm"), F.max("severity_norm"),
    F.min("funding_gap_norm"), F.max("funding_gap_norm"),
    F.min("scale_norm"), F.max("scale_norm")
).collect()[0]
print(f"   severity_norm range: [{score_norms[0]:.3f}, {score_norms[1]:.3f}] (expected: 0-1)")
print(f"   funding_gap_norm range: [{score_norms[2]:.3f}, {score_norms[3]:.3f}] (expected: 0-1)")
print(f"   scale_norm range: [{score_norms[4]:.3f}, {score_norms[5]:.3f}] (expected: 0-1)")
for i in range(0, 6, 2):
    if score_norms[i] is not None and (score_norms[i] < 0 or score_norms[i+1] > 1.001):
        failures.append(f"C2: Norm component out of [0,1] range")
        break
else:
    print(f"   ✅ All normalized components within [0,1]")

# Temporal series severity should also be 0-5
t5_range = t5.agg(F.min("severity_value"), F.max("severity_value")).collect()[0]
print(f"   Table 5 severity range: [{t5_range[0]}, {t5_range[1]}] (expected: 0-5)")
if t5_range[1] and t5_range[1] > 5.0:
    failures.append("C2: Temporal series severity exceeds 5.0")
else:
    print(f"   ✅ Temporal series within range")

# ──────────────────────────────────────────────────────────────────────
print(f"\n{'─' * 70}")
print("C3: JOIN FANOUT DETECTION")
print("    Did any joins multiply rows unexpectedly?")
print("─" * 70)

# Table 1 should have at most 1 row per iso3+year (the designed grain)
t1_dupes = t1.groupBy("iso3", "year").count().filter(F.col("count") > 1).count()
print(f"   Table 1 duplicate (iso3, year) pairs: {t1_dupes}")
if t1_dupes > 0:
    failures.append(f"C3: Table 1 has {t1_dupes} duplicate iso3+year rows (fanout)")
    print(f"   ❌ JOIN FANOUT DETECTED — investigate which join is causing duplication")
    # Show examples
    dupes = t1.groupBy("iso3", "year").count().filter(F.col("count") > 1).limit(3).collect()
    for d in dupes:
        print(f"     {d['iso3']} {d['year']}: {d['count']} rows")
else:
    print(f"   ✅ Table 1 grain is clean (one row per iso3+year)")

# Table 3 should also be iso3+year
t3_dupes = t3.groupBy("iso3", "year").count().filter(F.col("count") > 1).count()
print(f"   Table 3 duplicate (iso3, year) pairs: {t3_dupes}")
if t3_dupes > 0:
    failures.append(f"C3: Table 3 has {t3_dupes} duplicate rows")
else:
    print(f"   ✅ Table 3 grain is clean")

# Table 5 should be crisis_id + year_month
t5_dupes = t5.groupBy("crisis_id", "year_month").count().filter(F.col("count") > 1).count()
print(f"   Table 5 duplicate (crisis_id, year_month) pairs: {t5_dupes}")
if t5_dupes > 0:
    failures.append(f"C3: Table 5 has {t5_dupes} duplicate rows")
else:
    print(f"   ✅ Table 5 grain is clean")

# ──────────────────────────────────────────────────────────────────────
print(f"\n{'─' * 70}")
print("C4: CROSS-TABLE CONSISTENCY")
print("    Do derived tables agree with the source fact table?")
print("─" * 70)

# Table 4 (regional) should sum to Table 1 totals (for rows with region)
t1_total_req = t1.filter(F.col("region").isNotNull()).agg(F.sum("fts_requirements_usd")).collect()[0][0]
t4_total_req = t4.agg(F.sum("total_requirements_usd")).collect()[0][0]
if t1_total_req and t4_total_req:
    diff_pct = abs(t1_total_req - t4_total_req) / t1_total_req * 100
    print(f"   Table 1 total requirements (with region): ${t1_total_req:,.0f}")
    print(f"   Table 4 sum of regional requirements:     ${t4_total_req:,.0f}")
    print(f"   Difference: {diff_pct:.2f}%")
    if diff_pct > 1.0:
        failures.append(f"C4: Regional summary deviates {diff_pct:.1f}% from Table 1")
        print(f"   ❌ Mismatch exceeds 1% tolerance")
    else:
        print(f"   ✅ Regional totals match Table 1 (within 1%)")

# Table 3 row count should be <= Table 1 row count (it's a filtered subset)
if t3_count > t1_count:
    failures.append("C4: Table 3 has MORE rows than Table 1 (impossible)")
    print(f"   ❌ Table 3 ({t3_count}) > Table 1 ({t1_count})")
else:
    print(f"   ✅ Table 3 ({t3_count}) ≤ Table 1 ({t1_count}) as expected")

# ──────────────────────────────────────────────────────────────────────
print(f"\n{'─' * 70}")
print("C5: TEMPORAL CONSISTENCY")
print("    Are there unexpected gaps in time series data?")
print("─" * 70)

# Table 5: Check for gaps in monthly series per crisis
# A "gap" = a month where data should exist (between first and last observation)
# but is NULL. This is expected (sparse data) but we quantify it.
t5_coverage = (
    t5.groupBy("crisis_id")
    .agg(
        F.min("year_month").alias("first_obs"),
        F.max("year_month").alias("last_obs"),
        F.count("*").alias("actual_obs")
    )
    .withColumn("expected_months",
                F.months_between(F.col("last_obs"), F.col("first_obs")).cast("int") + 1)
    .withColumn("coverage_pct",
                F.round(F.col("actual_obs") / F.col("expected_months") * 100, 1))
)
coverage_stats = t5_coverage.agg(
    F.avg("coverage_pct"), F.min("coverage_pct"), F.max("coverage_pct")
).collect()[0]
print(f"   Avg temporal coverage per crisis: {coverage_stats[0]:.0f}%")
print(f"   Min/Max coverage: {coverage_stats[1]:.0f}% / {coverage_stats[2]:.0f}%")
print(f"   Crises with <50% coverage: {t5_coverage.filter(F.col('coverage_pct') < 50).count()}")
print(f"   (Low coverage expected for ‘-’ sentinel periods — informational only)")

# Table 1: Check year continuity per country
# Some countries may have gaps (e.g., present in 2020, absent in 2021, back in 2022)
t1_gaps = (
    t1.filter(F.col("fts_requirements_usd").isNotNull())
    .withColumn("prev_year", F.lag("year", 1).over(Window.partitionBy("iso3").orderBy("year")))
    .withColumn("year_gap", F.col("year") - F.col("prev_year"))
    .filter(F.col("year_gap") > 1)
)
gap_count = t1_gaps.count()
print(f"   Table 1 year-gaps (country present, then missing, then back): {gap_count}")
if gap_count > 0:
    print(f"   (These are real — countries enter/exit appeals. Not a bug.)")

# ──────────────────────────────────────────────────────────────────────
print(f"\n{'─' * 70}")
print("C6: TRANSFORMATION CORRECTNESS (spot-check)")
print("    Verify a known data point transforms correctly through the pipeline.")
print("─" * 70)

# Spot-check: Compare Table 1's AFG 2026 row against raw silver data
spot_iso3, spot_year = "AFG", 2026
silver_fts = spark.table(f"{CATALOG}.{SCHEMA}.{SILVER_PREFIX}fts_requirements_funding_global")
silver_afg = silver_fts.filter(
    (F.col("countrycode") == spot_iso3) &
    (F.col("year") == spot_year) &
    (F.col("requirements").isNotNull())
).agg(
    F.sum("requirements").alias("expected_req"),
    F.sum("funding").alias("expected_fund")
).collect()[0]

gold_afg = t1.filter(
    (F.col("iso3") == spot_iso3) & (F.col("year") == spot_year)
).select("fts_requirements_usd", "fts_funding_usd").collect()

if gold_afg and silver_afg["expected_req"]:
    req_match = abs(gold_afg[0]["fts_requirements_usd"] - silver_afg["expected_req"]) < 1.0
    fund_match = abs((gold_afg[0]["fts_funding_usd"] or 0) - (silver_afg["expected_fund"] or 0)) < 1.0
    print(f"   Spot check: {spot_iso3} {spot_year}")
    print(f"     Silver FTS requirements sum: ${silver_afg['expected_req']:,.0f}")
    print(f"     Gold Table 1 fts_requirements_usd: ${gold_afg[0]['fts_requirements_usd']:,.0f}")
    print(f"     Match: {'\u2705' if req_match else '\u274c'} requirements | {'\u2705' if fund_match else '\u274c'} funding")
    if not (req_match and fund_match):
        failures.append(f"C6: Spot check failed for {spot_iso3} {spot_year}")
else:
    print(f"   ⚠️ Could not spot-check (data not found)")

# ──────────────────────────────────────────────────────────────────────
print(f"\n{'─' * 70}")
print("C7: DATA LOSS DETECTION")
print("    Did we lose rows during transformation?")
print("─" * 70)

# FTS silver has 3,836 rows. After groupBy(countrycode, year) with requirements not null,
# we expect fewer rows (aggregated). But we should not lose COUNTRIES.
silver_countries = silver_fts.filter(F.col("requirements").isNotNull()).select("countrycode").distinct().count()
gold_countries_with_fts = t1.filter(F.col("fts_requirements_usd").isNotNull()).select("iso3").distinct().count()
print(f"   Silver FTS distinct countries (with requirements): {silver_countries}")
print(f"   Gold Table 1 distinct countries (with FTS data): {gold_countries_with_fts}")
if gold_countries_with_fts < silver_countries:
    lost = silver_countries - gold_countries_with_fts
    failures.append(f"C7: Lost {lost} countries during FTS transformation")
    print(f"   ❌ Lost {lost} countries")
else:
    print(f"   ✅ No country loss in FTS aggregation")

# HNO silver: distinct countries should match Table 2
silver_hno = spark.table(f"{CATALOG}.{SCHEMA}.{SILVER_PREFIX}hpc_hno")
hno_countries = silver_hno.filter(
    (F.col("admin_1_pcode").isNull()) & (F.col("cluster") != "ALL")
).select("country_iso3").distinct().count()
t2_countries = t2.select("iso3").distinct().count()
print(f"   Silver HNO countries (sector-level): {hno_countries}")
print(f"   Gold Table 2 countries: {t2_countries}")
if t2_countries < hno_countries:
    print(f"   ⚠️ Lost {hno_countries - t2_countries} countries (may be admin-level filtering)")
else:
    print(f"   ✅ No country loss in HNO transformation")

# =============================================================================
# FINAL SUMMARY
# =============================================================================
print(f"\n\n{'=' * 70}")
print("FINAL VALIDATION SUMMARY")
print("=" * 70)
print(f"\n   Part A: Data quality checks — see above")
print(f"   Part B verdicts:")
print(f"     B1 Score Discrimination: {verdict_b1}")
print(f"     B2 Face Validity:         {verdict_b2}")
print(f"     B3 Sector Join:           {verdict_b3}")
print(f"     B4 Temporal Features:     Persist={verdict_b4_persist} | Trend={verdict_b4_trend}")
print(f"     B5 Allocation Signal:     {verdict_b5}")
print(f"   Part C reliability failures: {len(failures)}")
if failures:
    print(f"   ❌ FAILURES:")
    for f_item in failures:
        print(f"     - {f_item}")
    print(f"\n   ACTION REQUIRED: Investigate C-failures before using gold tables.")
else:
    print(f"   ✅ ALL RELIABILITY CHECKS PASSED")

print(f"\n   Gold layer validation complete.")
print(f"   Tables ready: {GOLD_PREFIX}country_year_funding, sector_funding_gaps,")
print(f"                 overlooked_crises_scored, regional_summary, inform_temporal_series")